# DHB-XR: Features and Use-Cases Demo

This notebook demonstrates the main features and use-cases of **dhb_xr** (DHB Extended Representations) for SE(3) invariant trajectory encoding. Inspired by MATLAB Live Script workflows, we walk through:

1. **Constructing a 6-DOF trajectory** — position and orientation over time
2. **Encode and decode** — DHB-DR invariants and reconstruction
3. **SE(3) invariance** — invariants are unchanged under global rigid transforms
4. **DHB-DR vs DHB-QR** — Euler vs quaternion-based invariants
5. **DHB-TI (time-invariant)** — reparameterization by geometric progress
6. **Trajectory adaptation** — retargeting to a new initial pose
7. **Visualization** — 3D trajectories and invariant time series
8. **Optional: VLA tokenization** — discrete tokens for action representation

**Requirements:** `numpy`, `scipy`, `matplotlib`. Optional: `torch` for tokenization.

---
## Setup and imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

from dhb_xr import encode_dhb_dr, decode_dhb_dr
from dhb_xr.core.types import DHBMethod
from dhb_xr.core import geometry as geom
from dhb_xr.encoder.dhb_ti import compute_progress, resample_by_progress, encode_dhb_dr_ti
from dhb_xr.visualization.plot import plot_se3_trajectory, plot_invariants
from dhb_xr.optimization.casadi_solver import generate_trajectory

np.random.seed(42)
plt.rcParams['figure.figsize'] = (8, 5)

---
## 1. Construct a 6-DOF trajectory

We build a smooth SE(3) trajectory: **position** as a parametric curve (e.g. spiral or sinusoids) and **orientation** as time-varying Euler angles converted to quaternions (wxyz). This mimics typical motion data from robots or motion capture.

- **Position:** $x(t)$, $y(t)$, $z(t)$ — e.g. exponential + sinusoids
- **Orientation:** roll, pitch, yaw (Euler ZYX) → rotation matrix → quaternion (wxyz)

The result is `positions` (N×3) and `quaternions` (N×4) with **wxyz** convention (scalar first).

In [ ]:
def build_demo_trajectory(n=80, dt=0.05):
    """Build a 6-DOF trajectory: position curve + orientation over time."""
    t = np.linspace(0, (n - 1) * dt, n)
    # Position: smooth curve (inspired by mainDHB.m)
    x = 0.1 * np.exp(t * 0.3)
    y = 0.5 + 0.3 * np.sin(t * 2)
    z = 0.2 * np.cos(t)
    positions = np.column_stack([x, y, z])
    # Orientation: roll, pitch, yaw (rad) -> quaternion wxyz
    roll = 0.3 * np.sin(t)
    pitch = 0.2 * np.cos(t * 1.5)
    yaw = 0.1 * t
    quaternions = np.array([
        geom.rot_to_quat(
            geom.euler_to_rot(np.array([roll[i], pitch[i], yaw[i]]))
        ) for i in range(n)
    ])
    return positions, quaternions

positions, quaternions = build_demo_trajectory(n=80)
print("Trajectory length:", len(positions))
print("Positions shape:", positions.shape, "| Quaternions shape:", quaternions.shape)
print("First position:", positions[0])
print("First quaternion (wxyz):", quaternions[0])

### Plot: 3D position trajectory

The curve shows the Cartesian path of the rigid body. Optionally we can draw orientation frames at sampled points.

In [ ]:
fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection='3d')
plot_se3_trajectory(positions, quaternions, ax=ax, title="6-DOF trajectory (position + orientation)", show_orientation=True)
ax.legend()
plt.tight_layout()
plt.show()

---
## 2. Encode and decode with DHB-DR

**DHB-DR** (double-reflection) computes *invariants* from the trajectory: for each step we get **linear invariants** (magnitude + Euler angles of the linear frame step) and **angular invariants** (magnitude + Euler angles of the angular step). These describe the motion in a way that is **independent of the global reference frame** (see next section).

- **Encode:** `encode_dhb_dr(positions, quaternions, ...)` → `linear_motion_invariants` (N′×4), `angular_motion_invariants` (N′×4), `initial_pose`
- **Decode:** `decode_dhb_dr(linear_inv, angular_inv, initial_pose, ...)` → `positions`, `quaternions` (reconstructed trajectory)

With default padding, the encoder output length N′ is slightly larger than N−2 so that decoding with `drop_padded=True` aligns with the original length convention.

In [ ]:
out = encode_dhb_dr(
    positions, quaternions,
    method="pos",
    use_default_initial_frames=True,
    dhb_method=DHBMethod.DOUBLE_REFLECTION,
)
linear_inv = out["linear_motion_invariants"]
angular_inv = out["angular_motion_invariants"]
initial_pose = out["initial_pose"]

print("Encoded linear invariants shape:", linear_inv.shape)
print("Encoded angular invariants shape:", angular_inv.shape)
print("Initial pose position:", initial_pose["position"])

decoded = decode_dhb_dr(
    linear_inv, angular_inv, initial_pose,
    method="pos", dhb_method=DHBMethod.DOUBLE_REFLECTION, drop_padded=True,
)
pos_dec = decoded["positions"]
quat_dec = decoded["quaternions"]
print("Decoded positions shape:", pos_dec.shape)
print("Reconstruction error (position):", np.linalg.norm(pos_dec - positions[:len(pos_dec)]))
print("Reconstruction error (quat, geodesic approx):", np.mean([np.linalg.norm(geom.quat_relative_axis_angle(quat_dec[i], quaternions[i])) for i in range(len(quat_dec))]))

### Plot: Invariant time series

The **first column** of each invariant block is the **step magnitude** (translational and rotational); the remaining columns are orientation parameters (Euler angles in DHB-DR). These invariants fully describe the motion up to a global SE(3) frame.

In [ ]:
fig, axes = plt.subplots(2, 1, sharex=True, figsize=(10, 6))
plot_invariants(linear_inv, angular_inv, ax=axes, title="DHB-DR invariants (linear and angular)")
plt.tight_layout()
plt.show()

---
## 3. SE(3) invariance

A key property of DHB invariants is **invariance to global SE(3) transforms**. If we apply the same rigid transformation (rotation $R$ and translation $t$) to every pose of the trajectory, the *invariant sequence* (magnitudes and relative frame structure) is **unchanged**.

So: *encode(original)* and *encode(transformed)* yield the same linear/angular invariant **magnitudes** (first column). This is useful for comparing motions regardless of where they were performed or how the world frame was defined.

In [ ]:
def apply_se3(positions, quaternions, R, t):
    """Apply global SE(3): p' = R p + t, q' = R * q (rotation composition)."""
    p_new = (R @ positions.T).T + t
    q_new = np.array([geom.rot_to_quat(R @ geom.quat_to_rot(q)) for q in quaternions])
    return p_new, q_new

# Random SE(3) transform
axis = np.random.randn(3)
axis /= np.linalg.norm(axis)
R = geom.axis_angle_to_rot(axis * 0.5)
t_vec = np.array([0.2, -0.1, 0.15])
positions_tf, quaternions_tf = apply_se3(positions, quaternions, R, t_vec)

out_orig = encode_dhb_dr(positions, quaternions, method="pos", use_default_initial_frames=True, dhb_method=DHBMethod.DOUBLE_REFLECTION)
out_tf   = encode_dhb_dr(positions_tf, quaternions_tf, method="pos", use_default_initial_frames=True, dhb_method=DHBMethod.DOUBLE_REFLECTION)

lin_orig = out_orig["linear_motion_invariants"]
ang_orig = out_orig["angular_motion_invariants"]
lin_tf   = out_tf["linear_motion_invariants"]
ang_tf   = out_tf["angular_motion_invariants"]

match_lin = np.allclose(lin_orig[:, 0], lin_tf[:, 0])
match_ang = np.allclose(ang_orig[:, 0], ang_tf[:, 0])
print("Linear  magnitude invariant (first column) match:", match_lin)
print("Angular magnitude invariant (first column) match:", match_ang)
print("Max diff linear  mag:", np.max(np.abs(lin_orig[:, 0] - lin_tf[:, 0])))
print("Max diff angular mag:", np.max(np.abs(ang_orig[:, 0] - ang_tf[:, 0])))

### Plot: Original vs transformed trajectory

The two trajectories differ by a global rigid transform; their DHB invariant magnitudes are the same.

In [ ]:
fig = plt.figure(figsize=(9, 5))
ax = fig.add_subplot(111, projection='3d')
ax.plot(positions[:, 0], positions[:, 1], positions[:, 2], 'b-', label='Original', linewidth=2)
ax.plot(positions_tf[:, 0], positions_tf[:, 1], positions_tf[:, 2], 'r--', label='After SE(3) transform', linewidth=1.5)
ax.set_xlabel('x'); ax.set_ylabel('y'); ax.set_zlabel('z')
ax.set_title('Same motion in two frames: invariants match')
ax.legend()
plt.tight_layout()
plt.show()

---
## 4. DHB-DR vs DHB-QR

- **DHB-DR** uses **Euler angles** for the relative orientation at each step → 4 values per component (magnitude + 3 Euler). Can suffer from gimbal lock.
- **DHB-QR** uses **quaternions** for the relative rotation → 5 values per component (magnitude + 4D quaternion, with unit norm). Singularity-free and convenient for geodesic losses.

Same trajectory encoded with both yields different *shapes* (e.g. linear invariants (N,4) vs (N,5)); the *information* is equivalent up to representation.

In [ ]:
from dhb_xr.encoder.dhb_dr import encode_dhb_dr
from dhb_xr.encoder.dhb_qr import encode_dhb_qr

out_dr = encode_dhb_dr(positions, quaternions, method="pos", use_default_initial_frames=True, dhb_method=DHBMethod.DOUBLE_REFLECTION)
out_qr = encode_dhb_qr(positions, quaternions, method="pos", use_default_initial_frames=True)

print("DHB-DR linear invariants shape:", out_dr["linear_motion_invariants"].shape)
print("DHB-QR linear invariants shape:", out_qr["linear_motion_invariants"].shape)
print("DHB-DR: 4 values per step (magnitude + Euler). DHB-QR: 5 values per step (magnitude + quat).")

---
## 5. DHB-TI: Time-invariant reparameterization

Raw DHB invariants depend on **how densely** the trajectory is sampled in time: faster execution or lower frame rate changes step sizes and thus magnitudes. **DHB-TI** reparameterizes the trajectory by a **geometric progress variable** (e.g. arc-length or hybrid with rotation) and resamples at **uniform progress knots** before encoding. The resulting invariants are (approximately) **independent of execution speed and sampling rate**.

- **Progress:** translation (arc-length), angular (rotation magnitude), or **hybrid** $\sigma_{i+1} = \sigma_i + \alpha\|\Delta p_i\| + (1-\alpha)\|\Delta r_i\|$
- **Resample:** uniform $\sigma_k = k \Sigma/(M-1)$, interpolate positions (spline) and orientations (SLERP)
- **Encode:** apply DHB-DR (or DHB-QR) to the resampled sequence → `encode_dhb_dr_ti(...)`

In [ ]:
M = 40
progress = compute_progress(positions, quaternions, kind="hybrid", alpha=0.5)
pos_ti, quat_ti = resample_by_progress(positions, quaternions, M, progress_kind="hybrid", alpha=0.5)

print("Total progress (hybrid, α=0.5):", progress[-1])
print("Resampled trajectory length:", M)

out_ti = encode_dhb_dr_ti(positions, quaternions, M, progress_kind="hybrid", alpha=0.5,
                          method="pos", use_default_initial_frames=True, dhb_method=DHBMethod.DOUBLE_REFLECTION)
print("DHB-TI (DR) linear invariants shape:", out_ti["linear_motion_invariants"].shape)

### Plot: Original vs time-invariant resampled path

Resampled poses lie on the same geometric path but at uniform progress; encoding this sequence gives timing-robust invariants.

In [ ]:
fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection='3d')
ax.plot(positions[:, 0], positions[:, 1], positions[:, 2], 'b-', alpha=0.6, label='Original (time-sampled)')
ax.plot(pos_ti[:, 0], pos_ti[:, 1], pos_ti[:, 2], 'ro-', markersize=4, label=f'Resampled by progress (M={M})')
ax.set_xlabel('x'); ax.set_ylabel('y'); ax.set_zlabel('z')
ax.set_title('DHB-TI: Resample by geometric progress')
ax.legend()
plt.tight_layout()
plt.show()

---
## 6. Trajectory adaptation (retargeting)

We can **adapt** a demo trajectory to a **new initial pose** (and optionally goal): resample the demo, encode to invariants, then **decode** from the new initial pose. The motion *style* (invariants) is preserved while the start (and optionally end) pose is changed — useful for transferring a motion to a different robot or workspace.

`generate_trajectory(demo_pos, demo_quat, pose_init, pose_goal, traj_length, ...)` returns the adapted positions and quaternions.

In [ ]:
pose_init = {"position": positions[0].copy(), "quaternion": quaternions[0].copy()}
pose_goal = {
    "position": positions[-1].copy() + np.array([0.15, 0.1, 0.0]),
    "quaternion": quaternions[-1].copy(),
}
traj_length = 50
result = generate_trajectory(
    positions, quaternions,
    pose_init, pose_goal,
    traj_length=traj_length,
    smoothing=False,
    dhb_method=DHBMethod.DOUBLE_REFLECTION,
)
adapted_pos = result["adapted_pos_data"]
adapted_quat = result["adapted_quat_data"]
print("Adapted trajectory length:", len(adapted_pos))
print("First pose (target init):", adapted_pos[0])
print("Last pose (toward goal):", adapted_pos[-1])

In [ ]:
fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection='3d')
ax.plot(positions[:, 0], positions[:, 1], positions[:, 2], 'b-', alpha=0.7, label='Demo')
ax.plot(adapted_pos[:, 0], adapted_pos[:, 1], adapted_pos[:, 2], 'g-', alpha=0.7, label='Adapted (new init/goal)')
ax.scatter(*pose_init["position"], color='blue', s=80, marker='o', label='Init')
ax.scatter(*pose_goal["position"], color='green', s=80, marker='^', label='Goal')
ax.set_xlabel('x'); ax.set_ylabel('y'); ax.set_zlabel('z')
ax.set_title('Trajectory adaptation: same invariants, new boundaries')
ax.legend()
plt.tight_layout()
plt.show()

---
## 7. Optional: VLA tokenization (DHB-Token)

For **Vision-Language-Action** models, invariants can be **tokenized** with a causal VQ-VAE: the continuous invariant sequence is compressed into **discrete token indices** that can be generated autoregressively or used for retrieval. Here we encode the trajectory to invariants, run it through `DHBTokenizer`, and inspect token indices and reconstruction.

**Requires:** `torch`.

In [ ]:
try:
    import torch
    from dhb_xr.tokenization.vqvae import DHBTokenizer

    inv_combined = np.concatenate([linear_inv, angular_inv], axis=1).astype(np.float32)
    x = torch.from_numpy(inv_combined).unsqueeze(0)  # (1, T, 8)
    model = DHBTokenizer(invariant_dim=8, latent_dim=16, codebook_size=64)
    indices, reconstructed, z, z_q = model(x)
    print("Invariants shape:", x.shape)
    print("Token indices shape:", indices.shape)
    print("Reconstructed shape:", reconstructed.shape)
    print("Sample tokens (first 8):", indices[0, :8].tolist())
    rec_mse = torch.nn.functional.mse_loss(reconstructed, x).item()
    print("Reconstruction MSE:", rec_mse)
except ImportError as e:
    print("Skipping VLA tokenization (torch required):", e)

---
## Summary

| Feature | Use-case |
|--------|----------|
| **DHB-DR / DHB-QR** | Encode/decode SE(3) trajectories; invariants are frame-independent |
| **SE(3) invariance** | Compare motions across different reference frames |
| **DHB-TI** | Robustness to execution speed and sampling rate (reparameterize by progress) |
| **Trajectory adaptation** | Retarget a demo to new start/goal poses (same motion, new boundaries) |
| **Tokenization** | Discrete action representation for VLAs |

See the **examples/** scripts and **docs/examples_walkthrough.md** for more.